<a href="https://colab.research.google.com/github/FAB-V-V/Busca-minas/blob/main/Visualizacion_buscaminas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Visualización del Buscaminas
**Lógica para Ciencias de la Computación — 2026-II**
Juan Camilo Lozano González y Fabiana Vargas Velasco

Este notebook muestra cómo usar cada función del módulo `visualizacion_buscaminas.py`.

**Representación.** Un solo descriptor `Casilla = Descriptor([5, 5, 10])`. La letra $N_{x,y,k}$ significa *"la casilla $(x,y)$ tiene el valor $k$"*, donde $k \in \{0,\dots,8\}$ es el número de minas vecinas y $k = 9$ significa que hay una mina. $x$ es la fila (0 arriba) y $y$ la columna (0 a la izquierda).

## 0. Importar el módulo
Los archivos `Logica.py` y `visualizacion_buscaminas.py` deben estar en la misma carpeta que este notebook.

In [ ]:
from visualizacion_buscaminas import *

ModuleNotFoundError: No module named 'visualizacion_buscaminas'

## 1. El descriptor `Casilla`: de casilla a letra y de letra a casilla
- `Casilla.ravel([x, y, k])` convierte la terna en una **letra** (un solo carácter).
- `Casilla.unravel(letra)` hace lo contrario.

In [ ]:
letra = Casilla.ravel([0, 2, 9])     # "la casilla (0,2) tiene una mina"
print('Letra:', letra)
print('De vuelta:', [int(v) for v in Casilla.unravel(letra)])

NameError: name 'Casilla' is not defined

## 2. `interpretacion_desde_minas(minas)`: fabricar una interpretación a mano
Recibe la lista de casillas con mina y devuelve el diccionario `I = {letra: True/False}` **completo** (250 letras): calcula el número de cada casilla contando sus vecinas.
Sirve para probar el visualizador sin tener todavía un SAT-solver. Aquí usamos el tablero de ejemplo del paper (Tabla II).

In [ ]:
minas = [(0, 2), (1, 0), (1, 3), (2, 1), (3, 4), (4, 2)]
I = interpretacion_desde_minas(minas)
print('Número de letras:', len(I))
print('Letras verdaderas:', sum(I.values()))    # 25: una por casilla

NameError: name 'interpretacion_desde_minas' is not defined

Podemos leer qué letras quedaron verdaderas:

In [ ]:
for letra in I:
    if I[letra]:
        x, y, k = Casilla.unravel(letra)
        print(f'({x},{y}) ->', 'mina' if k == 9 else k)

## 3. `leer_tablero(I)`: traducir el diccionario a una tabla 5×5
Devuelve una lista de filas; en cada casilla queda la **lista** de valores $k$ cuya letra es verdadera.
Si todo está bien, cada casilla tiene exactamente un valor. Ignora las letras falsas y las que no son del descriptor (p. ej. las de Tseitin).

In [ ]:
tablero = leer_tablero(I)
for fila in tablero:
    print(fila)

## 4. `dibujar_tablero(I)`: dibujar el tablero
- `dibujar_tablero(I)` muestra la figura aquí.
- `dibujar_tablero(I, titulo='...')` le pone título.
- `dibujar_tablero(I, n=1)` en vez de mostrarla la guarda como `tablero_1.png`.

Devuelve la **lista de problemas** encontrados. Si sale `[]`, toda casilla tiene exactamente un valor.

In [ ]:
problemas = dibujar_tablero(I, titulo='Tabla II')
print('Problemas:', problemas)

Guardar a archivo:

In [ ]:
dibujar_tablero(I, n=1, titulo='Tabla II')   # crea tablero_1.png

## 5. Detección de errores
El visualizador marca dos situaciones que **no deberían ocurrir** si las reglas están bien:
- **"!" con borde rojo**: la casilla tiene más de un valor verdadero (viola la unicidad).
- **"?" en gris**: la casilla no tiene ningún valor verdadero (viola R1).

Las provocamos a mano para verlas:

In [ ]:
J = dict(I)                              # copia, para no dañar I
J[Casilla.ravel([0, 0, 9])] = True       # (0,0) ahora dice 1 y también mina
J[Casilla.ravel([4, 0, 0])] = False      # (4,0) se queda sin valor
print(dibujar_tablero(J, titulo='Con errores'))

## 6. Interpretaciones parciales
DPLL puede devolver un diccionario solo con algunas letras. Las que faltan se toman como falsas, así que basta con pasar las verdaderas:

In [ ]:
parcial = {letra: True for letra in I if I[letra]}
print('Letras en el diccionario parcial:', len(parcial))
dibujar_tablero(parcial, titulo='Interpretación parcial')

## 7. Cómo se usará con un SAT-solver
Cuando las reglas y el solver estén listos, el flujo será:
```python
I = solver(reglas)          # diccionario {letra: True/False}
if I is not None:
    dibujar_tablero(I)
else:
    print('¡No hay solución!')
```